In [ ]:
# Import Required Libraries

import warnings
warnings.filterwarnings("ignore")

import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.model_selection import (
    RandomizedSearchCV
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [32]:
# Load Processed Datasets

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

X_train_scaled = pd.read_csv(
    "../data/processed/X_train_scaled.csv"
)

X_test_scaled = pd.read_csv(
    "../data/processed/X_test_scaled.csv"
)

y_train = pd.read_csv("../data/processed/y_train.csv")

y_test = pd.read_csv("../data/processed/y_test.csv")

print("Processed Datasets Loaded Successfully")

Processed Datasets Loaded Successfully


In [33]:
# Dataset Verification

print("Training Feature Matrix :", X_train.shape)
print("Testing Feature Matrix  :", X_test.shape)

print()

print("Training Target :", y_train.shape)
print("Testing Target  :", y_test.shape)

Training Feature Matrix : (79913, 31)
Testing Feature Matrix  : (19979, 31)

Training Target : (79913, 1)
Testing Target  : (19979, 1)


In [ ]:
# Remove Identifier Columns

identifier_columns = [
    "Run",
    "Event"
]

X_train = X_train.drop(
    columns=identifier_columns,
    errors="ignore"
)

X_test = X_test.drop(
    columns=identifier_columns,
    errors="ignore"
)

X_train_scaled = X_train_scaled.drop(
    columns=identifier_columns,
    errors="ignore"
)

X_test_scaled = X_test_scaled.drop(
    columns=identifier_columns,
    errors="ignore"
)

print("Identifier Columns Removed Successfully")

Identifier Columns Removed Successfully


In [35]:
# Convert Target Variables to Series

y_train = y_train.squeeze()

y_test = y_test.squeeze()

print("Target Variables Converted Successfully")

Target Variables Converted Successfully


In [36]:
# Import Regression Models

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.tree import (
    DecisionTreeRegressor
)

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from xgboost import (
    XGBRegressor
)

from lightgbm import (
    LGBMRegressor
)

from catboost import (
    CatBoostRegressor
)

print("Regression Models Imported Successfully")

Regression Models Imported Successfully


In [37]:
# Initialize Regression Models

models = {

    "Linear Regression":
        LinearRegression(),

    "Ridge":
        Ridge(),

    "Lasso":
        Lasso(),

    "ElasticNet":
        ElasticNet(),

    "Decision Tree":
        DecisionTreeRegressor(
            random_state=42
        ),

    "Random Forest":
        RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ),

    "Extra Trees":
        ExtraTreesRegressor(
            random_state=42,
            n_jobs=-1
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            random_state=42
        ),

    "XGBoost":
        XGBRegressor(
            random_state=42,
            n_jobs=-1,
            verbosity=0
        ),

    "LightGBM":
        LGBMRegressor(
            random_state=42,
            verbosity=-1
        ),

    "CatBoost":
        CatBoostRegressor(
            random_state=42,
            verbose=0
        )

}

print("Regression Models Initialized Successfully")

Regression Models Initialized Successfully


In [38]:
# Evaluate Regression Model

def evaluate_model(

    model,
    X_train_data,
    X_test_data,
    y_train,
    y_test

):

    # Train the model
    model.fit(
        X_train_data,
        y_train
    )

    # Predict on the test set
    predictions = model.predict(
        X_test_data
    )

    # Evaluation Metrics
    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    n = X_test_data.shape[0]
    p = X_test_data.shape[1]

    adjusted_r2 = 1 - (
        (1 - r2) *
        (n - 1) /
        (n - p - 1)
    )

    return (
        mae,
        rmse,
        r2,
        adjusted_r2,
        predictions
    )

In [39]:
# Train Baseline Models

results = []

trained_models = {}

scaled_models = [

    "Linear Regression",
    "Ridge",
    "Lasso",
    "ElasticNet"

]

for model_name, model in models.items():

    # Select the appropriate dataset

    if model_name in scaled_models:

        X_train_data = X_train_scaled
        X_test_data = X_test_scaled

    else:

        X_train_data = X_train
        X_test_data = X_test

    # Evaluate model

    mae, rmse, r2, adjusted_r2, predictions = evaluate_model(

        model,
        X_train_data,
        X_test_data,
        y_train,
        y_test

    )

    # Store trained model

    trained_models[model_name] = model

    # Store evaluation metrics

    results.append({

        "Model": model_name,

        "Version": "Baseline",

        "MAE": mae,

        "RMSE": rmse,

        "R²": r2,

        "Adjusted R²": adjusted_r2

    })

print("Baseline Models Trained Successfully")

KeyboardInterrupt: 

In [ ]:
# Create Model Comparison Table

results_df = pd.DataFrame(
    results
)

results_df = results_df.sort_values(

    by="RMSE",

    ascending=True

).reset_index(
    drop=True
)

display(results_df)

In [ ]:
# Best Baseline Model

best_model_name = results_df.iloc[0]["Model"]

print("Best Baseline Model :", best_model_name)

In [ ]:
# Compare Models Using RMSE

plt.figure(figsize=(10, 6))

sns.barplot(

    data=results_df,

    x="RMSE",

    y="Model"

)

plt.title("RMSE Comparison of Regression Models")

plt.xlabel("RMSE")

plt.ylabel("Model")

plt.tight_layout()

plt.show()

In [ ]:
# Compare Models Using MAE

plt.figure(figsize=(10, 6))

sns.barplot(

    data=results_df,

    x="MAE",

    y="Model"

)

plt.title("MAE Comparison of Regression Models")

plt.xlabel("MAE")

plt.ylabel("Model")

plt.tight_layout()

plt.show()

In [ ]:
# Compare Models Using R²

plt.figure(figsize=(10, 6))

sns.barplot(

    data=results_df,

    x="R²",

    y="Model"

)

plt.title("R² Comparison of Regression Models")

plt.xlabel("R²")

plt.ylabel("Model")

plt.tight_layout()

plt.show()

In [ ]:
# Compare Models Using Adjusted R²

plt.figure(figsize=(10, 6))

sns.barplot(

    data=results_df,

    x="Adjusted R²",

    y="Model"

)

plt.title("Adjusted R² Comparison of Regression Models")

plt.xlabel("Adjusted R²")

plt.ylabel("Model")

plt.tight_layout()

plt.show()

In [ ]:
# Save Baseline Model Results

os.makedirs(

    "../results",

    exist_ok=True

)

results_df.to_csv(

    "../results/baseline_model_results.csv",

    index=False

)

# Evaluation Metrics
#
# MAE  : Average prediction error. Lower values indicate better performance.
# RMSE : Similar to MAE but gives more importance to larger errors.
# R²   : Measures how well the model explains the target variable. Higher is better.
# Adjusted R² : Modified R² that accounts for the number of input features.

print("Baseline Results Saved Successfully")

In [ ]:
# Define Hyperparameter Search Space

parameter_grids = {

    "Extra Trees": {

        "n_estimators": [100, 200, 300],

        "max_depth": [10, 20, 30, None],

        "min_samples_split": [2, 5, 10],

        "min_samples_leaf": [1, 2, 4]

    },

    "LightGBM": {

        "n_estimators": [100, 200, 300],

        "learning_rate": [0.01, 0.05, 0.1],

        "max_depth": [-1, 10, 20],

        "num_leaves": [31, 50, 100]

    },

    "XGBoost": {

        "n_estimators": [100, 200, 300],

        "learning_rate": [0.01, 0.05, 0.1],

        "max_depth": [3, 5, 7],

        "subsample": [0.8, 1.0],

        "colsample_bytree": [0.8, 1.0]

    }

}

In [ ]:
# Tune Selected Models

best_models = {}

tuned_results = []

models_to_tune = [

    "Extra Trees",

    "LightGBM",

    "XGBoost"

]

# Create a Sample for Hyperparameter Tuning

X_train_tune = X_train.sample(

    n=30000,

    random_state=42

)

y_train_tune = y_train.loc[

    X_train_tune.index

]

for model_name in models_to_tune:

    print(f"Tuning {model_name}...")

    model = models[model_name]

    search = RandomizedSearchCV(

        estimator=model,

        param_distributions=parameter_grids[model_name],

        n_iter=8,

        scoring="neg_root_mean_squared_error",

        cv=3,

        random_state=42,

        n_jobs=1

    )

    search.fit(

        X_train_tune,

        y_train_tune

    )

    best_model = search.best_estimator_

    predictions = best_model.predict(

        X_test

    )

    mae = mean_absolute_error(

        y_test,

        predictions

    )

    rmse = np.sqrt(

        mean_squared_error(

            y_test,

            predictions

        )

    )

    r2 = r2_score(

        y_test,

        predictions

    )

    n = X_test.shape[0]

    p = X_test.shape[1]

    adjusted_r2 = 1 - (

        (1 - r2) *

        (n - 1) /

        (n - p - 1)

    )

    best_models[model_name] = best_model

    tuned_results.append({

        "Model": model_name,

        "Version": "Tuned",

        "MAE": mae,

        "RMSE": rmse,

        "R²": r2,

        "Adjusted R²": adjusted_r2

    })

print("Hyperparameter Tuning Completed")

In [ ]:
# Tuned Model Comparison

tuned_results_df = pd.DataFrame(

    tuned_results

)

tuned_results_df = tuned_results_df.sort_values(

    by="RMSE"

).reset_index(

    drop=True

)

display(

    tuned_results_df

)

In [ ]:
# Save Tuned Model Results

tuned_results_df.to_csv(

    "../results/tuned_model_results.csv",

    index=False

)

print("Tuned Results Saved Successfully")

In [ ]:
# Compare Baseline and Tuned Models

comparison_df = pd.concat(

    [

        results_df,

        tuned_results_df

    ],

    ignore_index=True

)

comparison_df = comparison_df.sort_values(

    by="RMSE"

).reset_index(

    drop=True

)

display(

    comparison_df
)

In [ ]:
# Display Overall Best Model

best_model_name = comparison_df.iloc[0]["Model"]

best_model_metrics = comparison_df.iloc[0]

print(f"Best Overall Model : {best_model_name}")
print(f"RMSE : {best_model_metrics['RMSE']:.3f}")
print(f"MAE : {best_model_metrics['MAE']:.3f}")
print(f"R² : {best_model_metrics['R²']:.3f}")

In [ ]:
# Retrieve Overall Best Model

if best_model_name in best_models:

    best_model = best_models[best_model_name]

else:

    best_model = trained_models[best_model_name]

print("Best Model Retrieved Successfully")

In [ ]:
# Perform Cross Validation

if best_model_name in [

    "Linear Regression",

    "Ridge",

    "Lasso",

    "ElasticNet"

]:

    X_cv = X_train_scaled

else:

    X_cv = X_train

cv_scores = cross_val_score(

    best_model,

    X_cv,

    y_train,

    cv=5,

    scoring="r2",

    n_jobs=-1

)

print("Cross Validation R² Scores")

print(cv_scores)

print()

print(f"Mean R² : {cv_scores.mean():.4f}")

print(f"Standard Deviation : {cv_scores.std():.4f}")

In [ ]:
# Generate Predictions Using Best Model

if best_model_name in [

    "Linear Regression",

    "Ridge",

    "Lasso",

    "ElasticNet"

]:

    predictions = best_model.predict(

        X_test_scaled

    )

else:

    predictions = best_model.predict(

        X_test

    )

residuals = y_test - predictions

In [ ]:
# Actual vs Predicted Values

plt.figure(figsize=(8,6))

plt.scatter(

    y_test,

    predictions,

    alpha=0.5

)

plt.plot(

    [

        y_test.min(),

        y_test.max()

    ],

    [

        y_test.min(),

        y_test.max()

    ],

    color="red"

)

plt.title("Actual vs Predicted")

plt.xlabel("Actual")

plt.ylabel("Predicted")

plt.tight_layout()

plt.show()

In [ ]:
# Residual Plot

plt.figure(figsize=(8,6))

plt.scatter(

    predictions,

    residuals,

    alpha=0.5

)

plt.axhline(

    0,

    color="red"

)

plt.title("Residual Plot")

plt.xlabel("Predicted")

plt.ylabel("Residual")

plt.tight_layout()

plt.show()

In [ ]:
# Residual Distribution

plt.figure(figsize=(8,6))

sns.histplot(

    residuals,

    bins=40,

    kde=True

)

plt.title("Residual Distribution")

plt.xlabel("Residual")

plt.tight_layout()

plt.show()

In [ ]:
# Save Best Model

os.makedirs(

    "../models",

    exist_ok=True

)

joblib.dump(

    best_model,

    "../models/best_model.pkl"

)

print("Best Model Saved Successfully")

In [ ]:
# Save Best Model Summary

best_model_summary = pd.DataFrame({

    "Model": [

        best_model_name

    ],

    "MAE": [

        comparison_df.iloc[0]["MAE"]

    ],

    "RMSE": [

        comparison_df.iloc[0]["RMSE"]

    ],

    "R²": [

        comparison_df.iloc[0]["R²"]

    ],

    "Adjusted R²": [

        comparison_df.iloc[0]["Adjusted R²"]

    ]

})

best_model_summary.to_csv(

    "../results/best_model_summary.csv",

    index=False

)

print("Best Model Summary Saved Successfully")